# log10-Resistivity

In [2]:
"""
ERT Batch Plotter — f001_res.dat to f044_res.dat
Produces one PNG per group of 4 files (4 panels per figure).
Output PNGs are saved in a subfolder: ert_layered_outputs/

Usage:
    python ERT_batch_f001_to_f044.py

Edit DATA_DIR and OUT_DIR below to match your paths.
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colorbar import ColorbarBase
from scipy.interpolate import griddata
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# USER SETTINGS — edit these paths
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR = Path(r"C:\Users\owner\OneDrive - huji.ac.il\שולחן העבודה\ERT-RWU-ERT-Wheat-project\Inversion_to_RWU_MultiGenotype_Analysis\raw_data\inversion_res_dat")
OUT_DIR  = DATA_DIR.parent / "ert_layered_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# File indices to process (f001 → f044)
FILE_INDICES = list(range(1, 45))   # 1, 2, 3, … 44

# How many panels per figure (4 recommended)
PANELS_PER_FIG = 4

# Genotype separator x-positions and labels
GENOTYPE_LINES  = [16.0, 21.5]
GENOTYPE_LABELS = [("IPLR-760", 13.0), ("WM-140", 18.75), ("WM-203", 24.25)]

# Colormap and resistivity range
CMAP  = plt.cm.viridis
VMIN  = 0.1
VMAX  = 1.8

# Spatial extent and grid resolution
X_MIN, X_MAX = 10, 27
Z_MIN, Z_MAX = -1.25, 0.0
GRID_NX = 1400
GRID_NZ = 350

DPI = 600
# ══════════════════════════════════════════════════════════════════════════════

# ── Pre-build interpolation grid ─────────────────────────────────────────────
xi = np.linspace(X_MIN, X_MAX, GRID_NX)
zi = np.linspace(Z_MIN, Z_MAX, GRID_NZ)
XI, ZI = np.meshgrid(xi, zi)

levels_fill = np.linspace(VMIN, VMAX, 81)


def load_and_interpolate(fpath):
    """Load a .dat file and interpolate log10-resistivity onto the grid."""
    d = np.loadtxt(fpath)
    mask = (
        (d[:, 0] >= X_MIN - 0.5) & (d[:, 0] <= X_MAX + 0.5) &
        (d[:, 1] >= Z_MIN - 0.1) & (d[:, 1] <= 0.05)
    )
    x, z, logr = d[mask, 0], d[mask, 1], d[mask, 3]

    grid_lin = griddata((x, z), logr, (XI, ZI), method='linear')
    grid_nn  = griddata((x, z), logr, (XI, ZI), method='nearest')
    grid     = np.where(np.isnan(grid_lin), grid_nn, grid_lin)
    return np.clip(grid, VMIN, VMAX)


def make_figure(panel_files, fig_index):
    """
    panel_files : list of (label, Path) tuples, length ≤ PANELS_PER_FIG
    fig_index   : integer used in the output filename
    """
    n = len(panel_files)
    fig, axes = plt.subplots(
        n, 1,
        figsize=(20, 3.5 * n),
        gridspec_kw={
            'hspace': 0.65,
            'left': 0.07, 'right': 0.88,
            'top': 0.96,  'bottom': 0.07
        }
    )
    if n == 1:
        axes = [axes]   # keep iterable

    fig.patch.set_facecolor('white')

    for i, (ax, (label, fpath)) in enumerate(zip(axes, panel_files)):
        print(f"  Plotting {fpath.name} …")
        grid = load_and_interpolate(fpath)

        # ── Filled contours + contour lines ───────────────────────────────
        ax.set_facecolor('white')
        ax.contourf(XI, ZI, grid, levels=levels_fill,
                    cmap=CMAP, vmin=VMIN, vmax=VMAX, extend='both')
        ax.contour(XI, ZI, grid, levels=14,
                   colors='black', linewidths=0.5, alpha=0.35)

        ax.set_xlim(X_MIN, X_MAX)
        ax.set_ylim(Z_MIN, Z_MAX)

        # ── Genotype dividers ─────────────────────────────────────────────
        for gx in GENOTYPE_LINES:
            ax.axvline(gx, color='white', linestyle='--',
                       linewidth=1.2, alpha=0.9)

        # ── Genotype labels (first panel of every figure only) ────────────
        if i == 0:
            for gname, lx in GENOTYPE_LABELS:
                ax.text(lx, 0.06, gname, color='red', fontsize=16,
                        fontweight='bold', ha='center', va='bottom',
                        transform=ax.transData, clip_on=False)

        # ── Panel label (file name / date) ────────────────────────────────
        ax.text(X_MIN + 0.15, -0.03, label, color='black', fontsize=13,
                va='top', ha='left', style='italic',
                transform=ax.transData)

        # ── Axes formatting ───────────────────────────────────────────────
        ax.tick_params(colors='black', labelsize=12)
        for spine in ax.spines.values():
            spine.set_edgecolor('#aaaaaa')

        ax.set_ylabel("Depth (m)", color='black', fontsize=13)
        ax.set_yticks([-1.0, -0.5, 0.0])
        ax.set_yticklabels(['-1.0', '-0.5', '0'], color='black', fontsize=12)

        if i == n - 1:   # bottom panel gets x-axis label
            ax.set_xlabel("Distance (m)", color='black', fontsize=13)
            ax.set_xticks(range(X_MIN, X_MAX + 1, 2))
            ax.set_xticklabels(
                [str(v) for v in range(X_MIN, X_MAX + 1, 2)],
                color='black', fontsize=12
            )
        else:
            ax.set_xticks(range(X_MIN, X_MAX + 1, 2))
            ax.set_xticklabels([])

    # ── Shared colorbar ───────────────────────────────────────────────────────
    cbar_ax = fig.add_axes([0.905, 0.07, 0.018, 0.89])
    norm = mcolors.Normalize(vmin=VMIN, vmax=VMAX)
    cb = ColorbarBase(cbar_ax, cmap=CMAP, norm=norm, orientation='vertical')
    cb.set_ticks([0.1, 0.4, 0.7, 1.0, 1.3, 1.6, 1.8])
    cb.ax.tick_params(colors='black', labelsize=12)
    cb.set_label('log$_{10}$ (Ω·m)', color='black', fontsize=13,
                 rotation=270, labelpad=20)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='black', fontsize=12)
    for spine in cb.ax.spines.values():
        spine.set_edgecolor('#aaaaaa')

    # ── Save ──────────────────────────────────────────────────────────────────
    first = panel_files[0][1].stem   # e.g. f001_res
    last  = panel_files[-1][1].stem  # e.g. f004_res
    out   = OUT_DIR / f"ERT_{first}_to_{last}_600dpi.png"
    fig.savefig(out, dpi=DPI, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  ✓ Saved → {out}")
    return out


# ── Main loop ─────────────────────────────────────────────────────────────────
def main():
    # Build list of (label, path) for every requested file index
    all_panels = []
    for idx in FILE_INDICES:
        fname = f"f{idx:03d}_res.dat"
        fpath = DATA_DIR / fname
        if fpath.exists():
            all_panels.append((fname.replace("_res.dat", ""), fpath))
        else:
            print(f"  ⚠  Not found, skipping: {fpath}")

    if not all_panels:
        print("No files found — check DATA_DIR path.")
        return

    print(f"Found {len(all_panels)} files. "
          f"Generating figures ({PANELS_PER_FIG} panels each) …\n")

    # Split into groups of PANELS_PER_FIG
    groups = [all_panels[i:i + PANELS_PER_FIG]
              for i in range(0, len(all_panels), PANELS_PER_FIG)]

    for fig_idx, group in enumerate(groups, start=1):
        print(f"Figure {fig_idx}/{len(groups)}: "
              f"{group[0][0]} → {group[-1][0]}")
        make_figure(group, fig_idx)

    print(f"\nAll done! Output folder: {OUT_DIR}")


if __name__ == "__main__":
    main()

Found 44 files. Generating figures (4 panels each) …

Figure 1/11: f001 → f004
  Plotting f001_res.dat …
  Plotting f002_res.dat …
  Plotting f003_res.dat …
  Plotting f004_res.dat …
  ✓ Saved → C:\Users\owner\OneDrive - huji.ac.il\שולחן העבודה\ERT-RWU-ERT-Wheat-project\Inversion_to_RWU_MultiGenotype_Analysis\raw_data\ert_layered_outputs\ERT_f001_res_to_f004_res_600dpi.png
Figure 2/11: f005 → f008
  Plotting f005_res.dat …
  Plotting f006_res.dat …
  Plotting f007_res.dat …
  Plotting f008_res.dat …
  ✓ Saved → C:\Users\owner\OneDrive - huji.ac.il\שולחן העבודה\ERT-RWU-ERT-Wheat-project\Inversion_to_RWU_MultiGenotype_Analysis\raw_data\ert_layered_outputs\ERT_f005_res_to_f008_res_600dpi.png
Figure 3/11: f009 → f012
  Plotting f009_res.dat …
  Plotting f010_res.dat …
  Plotting f011_res.dat …
  Plotting f012_res.dat …
  ✓ Saved → C:\Users\owner\OneDrive - huji.ac.il\שולחן העבודה\ERT-RWU-ERT-Wheat-project\Inversion_to_RWU_MultiGenotype_Analysis\raw_data\ert_layered_outputs\ERT_f009_res_to_